# Lexsi Track 3 — reproduce this entry in one click

**The CPU cells run on a free runtime: ~5 minutes, no GPU, no Hugging Face token.** They clone the
repo, run all 29 offline checks (the pytest suite, dataset SHA-256 pins, oracle, leak checks, reward,
stats, AuditKIT adapter, demo handlers, guard sandbox), re-hash every file in `docs/MANIFEST.md`
straight from the Hub, print the guard tables and the 8-payload adaptive probe, then download the
run logs and regenerate `docs/STATS.md` — every interval, paired test and the pooled three-seed
rate — failing loudly if the file changes.

**The last two cells need a GPU** (Runtime -> Change runtime type -> T4 GPU, ~4 minutes) and load the
shipped `sft4_adapter` to run one held-out episode and score it.

Last checked 2026-11-12. Every number below comes from a `summary.json` uploaded to
`primce/lexsi-runs`; nothing is estimated.


In [ ]:
!rm -rf lexsi-agentic-hinglish
!git clone -q https://github.com/PRINCE2-AI/lexsi-agentic-hinglish
%cd lexsi-agentic-hinglish
!git log --oneline -1

## 1. Offline checks (CPU)

`verify_all.py --with-tests` rebuilds every split, checks the SHA-256 pins, runs the oracle, runs the
pytest suite and imports the AuditKIT adapter without needing AuditKIT installed. Expected:
`all 29 checks passed`.


In [ ]:
!python scripts/verify_all.py --with-tests


## 1b. Provenance: re-hash every published file (CPU, network, no token)

`docs/MANIFEST.md` lists the sha256 of every published run summary, the shipped adapter weights and
the eight pinned eval files, plus one freeze digest over all of them. This re-downloads them from
`primce/lexsi-runs` and `primce/tiny-aya-hinglish-lora` and fails on any drift.


In [ ]:
!python scripts/manifest.py --check


## 1c. The guard, and a probe written against it (CPU)

`guard_eval.py` reproduces the detection table: **166/166** published payloads, **12/12** dev,
**5/8** blind, **0/294** clean episodes flagged. `adaptive_probe.py` measures eight payloads written
after reading `hinglish_agent/guard.py`: **3/8** on the text rules, **6/8** once the structural
allowlist drops the `note` key, two misses that spell the amount out.


In [ ]:
!python scripts/guard_eval.py
!python scripts/adaptive_probe.py


## 2. The oracle and the frozen splits

The oracle solves the 30-task hard split at the oracle call count. The tuned agent is scored the
same way: final ledger state, answer content, and call-level diagnostics — no LLM judge.

| Model | hard-30 | hard-90 |
|---|---:|---:|
| Base Tiny Aya, prompt-only | 3.33% (1/30) | 3.33% (3/90) |
| **Shipped `sft4_adapter`** | **96.67% (29/30)** | **93.33% (84/90)** |

Paired exact McNemar, same task ids: base → shipped is 81 fixed / 0 broken on hard-90 (p < 1e-20)
and 28 fixed / 0 broken on hard-30 (p = 7.5e-9). The same recipe on seeds 7 and 123 scores 84/90 and
83/90, so the pooled estimate for the recipe is **251/270 = 93.0%** on hard-90.

In [ ]:
!python scripts/run_oracle_eval.py --tasks data/tasks_hard_eval.jsonl --out runs/oracle_notebook

## 3. Recompute the published intervals (CPU, ~1 minute)

`runs/` is gitignored, so the per-task rows live in the public `primce/lexsi-runs` dataset. This
downloads them, regenerates `docs/STATS.md`, and then checks the file did not change: every Wilson
and bootstrap interval, the paired McNemar tests and the pooled three-seed rate are recomputed from
the rows instead of copied from the write-up.

In [ ]:
!python scripts/fetch_runs.py
!python scripts/bootstrap.py > /dev/null
!git diff --quiet docs/STATS.md && echo "PASS: docs/STATS.md regenerates byte-for-byte from the run logs" || (echo "MISMATCH:"; git diff docs/STATS.md)

## 4. One tuned episode (GPU)

Switch the runtime to a T4 GPU first. This loads the **shipped** adapter from the Hub and runs the
first frozen hard task end to end, then prints the ledger and the deterministic score.

In [ ]:
!nvidia-smi -L || echo "No GPU. Runtime -> Change runtime type -> T4 GPU, then rerun this cell."


In [ ]:
!pip install -q "transformers>=5.15" peft accelerate bitsandbytes

import json, sys, torch
sys.path.insert(0, ".")
from hinglish_agent.runner import run_episode, score_episode
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

assert torch.cuda.is_available(), "Switch the Colab runtime to T4 GPU and rerun."

tok = AutoTokenizer.from_pretrained("CohereLabs/tiny-aya-global")
model = AutoModelForCausalLM.from_pretrained(
    "CohereLabs/tiny-aya-global", torch_dtype=torch.float16, device_map="auto"
)
model = PeftModel.from_pretrained(
    model, "primce/tiny-aya-hinglish-lora", subfolder="sft4_adapter"
).eval()

task = json.loads(open("data/tasks_hard_eval.jsonl", encoding="utf-8").readline())

def call(messages):
    prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok(prompt, return_tensors="pt", truncation=True, max_length=4096).to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=256, do_sample=False,
            pad_token_id=tok.pad_token_id or tok.eos_token_id,
        )
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

episode = run_episode(call, task, max_steps=8)
print("TASK:", task["task"])
print("LEDGER:", json.dumps(episode["ledger"], indent=2))
print("EXPECTED:", json.dumps(task["expected_balances"], indent=2))
print("SCORE:", json.dumps(score_episode(episode, task), indent=2))

## 5. Training commands (GPU, ~2 h per stage on one T4)

These are commented out on purpose: they are the exact commands behind the frozen numbers. The
data mix is `data/tasks_train_mix3.jsonl` (374 rows: easy + hard + abstention + contrast), seed 42.

In [ ]:
# python scripts/train_sft.py --tasks data/tasks_train_mix3.jsonl --epochs 2 --seed 42 #     --out runs/sft4 --merge-out runs/sft4_merged
# python scripts/run_baseline.py --model runs/sft4_merged #     --tasks data/tasks_hard90_eval.jsonl --no-4bit --out runs/sft4_hard90
# python scripts/stats.py --a runs/sft3_hard90 --b runs/sft4_hard90
# python scripts/auditkit_eval.py --run sft4_hard90 --tasks data/tasks_hard90_eval.jsonl

# GRPO (AgentTune) needs Python 3.12: the two runs below both came out flat and are reported as such.
# python scripts/train_grpo.py --model runs/sft4_merged --tasks data/tasks_train_grpo.jsonl #     --out runs/grpo4 --max-steps 24 --generations 4 --lr 3e-6 --efficiency


## 6. What to read next

- Write-up: https://huggingface.co/spaces/primce/hinglish-track3-article
- Every held-out episode, scored, static and GPU-free: https://huggingface.co/spaces/primce/hinglish-episodes
- Live demo with the base/tuned switch, an injection scenario, and a guard panel a judge can paste
  their own tool result into: https://huggingface.co/spaces/primce/hinglish-expense-agent
- Adapter: https://huggingface.co/primce/tiny-aya-hinglish-lora (`sft4_adapter`)
- Run logs: https://huggingface.co/datasets/primce/lexsi-runs

Honest limits, all in the write-up: the 90-task holdout is still a probe (Wilson 86.2-96.9%, pooled
three-seed 251/270 = 93.0%), GRPO moved nothing in two runs, the guard is 5/8 on a blind payload set
and 6/8 on a probe written after reading its source, and is not shipped as a security boundary on
its own, and abstention is 4/8 on the shipped adapter (7/8 with the hardened prompt) against a
pre-registered bar of 6/8.
